# LightGBM Model

## Imports

In [1]:
import sys; sys.path.append("..")

from src.training import evaluate_lgbm
from src.evaluate import score_table

from src.features import build_holiday_lookup
from src.training import build_features

import pandas as pd
import numpy as np

In [2]:
train = pd.read_parquet("../data/processed/train.parquet")
stores = pd.read_parquet("../data/processed/stores.parquet")
oil = pd.read_parquet("../data/processed/oil.parquet")
holidays = pd.read_parquet("../data/processed/holidays.parquet")

## Validate Feature Creation Pipeline

In [3]:
national, local = build_holiday_lookup(holidays)
feats = build_features(train.copy(), stores, oil, national, local)

print("Shape:", feats.shape)
print("\nColumns:\n", list(feats.columns))
feats.head(20)

Shape: (3000888, 27)

Columns:
 ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'oil_roll_14', 'dow', 'month', 'day', 'year', 'payday', 'is_holiday', 'sales_lag_16', 'sales_lag_21', 'sales_lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_14', 'roll_std_14', 'roll_mean_30', 'roll_std_30']


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,...,is_holiday,sales_lag_16,sales_lag_21,sales_lag_28,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_30,roll_std_30
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8910,8910,2013-01-06,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10692,10692,2013-01-07,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12474,12474,2013-01-08,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14256,14256,2013-01-09,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16038,16038,2013-01-10,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
mask = (feats["store_nbr"] == 1).to_numpy() & (feats["family"] == "PRODUCE").to_numpy()
s = feats[mask].sort_values("date").reset_index(drop=True)

print("Sales at row 14:", s.loc[14, "sales"])
print("lag_16 at row 30:", s.loc[30, "sales_lag_16"])

assert s.loc[30, "sales_lag_16"] == s.loc[14, "sales"], "LAG MISALIGNED - Leak Risk"
print("lag_16 correctly references 16 rows back.")

Sales at row 14: 0.0
lag_16 at row 30: 0.0
lag_16 correctly references 16 rows back.


In [5]:
firsts = feats.sort_values("date").groupby(["store_nbr", "family"], observed=True).head(1)

print("First-row lag_16 all NaN?", firsts["sales_lag_16"].isna().all())
assert firsts["sales_lag_16"].isna().all(), "LEAK: lgas crossing series boundaries."

First-row lag_16 all NaN? True


In [6]:
for c in ["family", "city", "state", "type", "cluster"]:
    if c in feats.columns:
        print(f"{c}: {feats[c].dtype}")

print(feats.select_dtypes(include="object").columns.tolist()) # should be empty

family: category
city: category
state: category
type: category
cluster: int8
[]


In [7]:
drop = {"id", "date", "sales", "store_nbr"}
feature_cols = [c for c in feats.columns if c not in drop]
print("Features going to LightGBM:\n", feature_cols)

nan_pct = feats[feature_cols].isna().mean().sort_values(ascending=False)
print("\nNaN % by feature:\n", (nan_pct[nan_pct>0]*100).round(1))

Features going to LightGBM:
 ['family', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'oil_roll_14', 'dow', 'month', 'day', 'year', 'payday', 'is_holiday', 'sales_lag_16', 'sales_lag_21', 'sales_lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_14', 'roll_std_14', 'roll_mean_30', 'roll_std_30']

NaN % by feature:
 dcoilwtico      28.6
oil_roll_14     28.6
roll_std_30      2.7
roll_mean_30     2.7
roll_std_14      1.7
roll_mean_14     1.7
sales_lag_28     1.7
roll_std_7       1.3
roll_mean_7      1.3
sales_lag_21     1.2
sales_lag_16     1.0
dtype: float64


## TEST: Train Single Fold

In [8]:
lgbm_preds = evaluate_lgbm(train, stores, oil, holidays, params={"num_boost_round": 50})
print(lgbm_preds.shape)
print(lgbm_preds.head())
print("\nNulls in predictions?", lgbm_preds.y_pred.isna().sum())
print("Any negative predictions?", (lgbm_preds.y_pred < 0).sum())
print("y_pred range:", lgbm_preds.y_pred.min(), "-", lgbm_preds.y_pred.max())

(114048, 6)
   store_nbr      family       date  y_true    y_pred  fold
0          1  AUTOMOTIVE 2017-06-13     5.0  2.740517     0
1          1  AUTOMOTIVE 2017-06-14     5.0  3.039113     0
2          1  AUTOMOTIVE 2017-06-15     3.0  2.808556     0
3          1  AUTOMOTIVE 2017-06-16     4.0  2.693902     0
4          1  AUTOMOTIVE 2017-06-17     3.0  3.661100     0

Nulls in predictions? 0
Any negative predictions? 0
y_pred range: 0.2921800062074796 - 5351.449018154668


## Train and Evaluate LightGBM 

In [ ]:
lgbm_preds = evaluate_lgbm(train, stores, oil, holidays)
lgbm_overall, lgbm_family = score_table(lgbm_preds, "lightgbm")